# 01 · The Spam Filter That Taught Itself
### A story about rules, mistakes, and why machine learning exists

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/YOUR-REPO/blob/main/notebooks/01_rules_vs_learning.ipynb)

Let's go back in time. It's **1999**. The internet is booming, everyone's getting an email address, and right behind them come the spammers. Inboxes are filling up with junk: miracle pills, "make money fast" schemes, and prizes nobody actually won.

You've just landed your first job as a programmer, and your manager has one task for you: **stop the spam.**

So what do you do? You do what every programmer of that era did. You write rules. Back then, spam filters weren't clever at all. They were built from three simple ingredients:

- **Keyword lists:** if a message contains a word like "Viagra", block it.
- **Blocklists:** keep a list of known spammers and block anything they send.
- **IP address rules:** block messages that come from servers known to pump out spam.

Rigid, simple, and written entirely by hand. That's how spam filtering worked in the late 1990s.

In this notebook, **you're going to step into those shoes.** You'll build a 1999-style spam filter yourself, out of plain Python `if` statements, and test it on 5,572 real messages. You'll hit every wall those programmers hit. Then we'll fast-forward to **2002**, when one essay changed how the world fought spam, and you'll see machine learning do your job better than you could.

> **A quick note on the data.** We don't have a pile of 1999 emails, but we have the next best thing: a collection of real **SMS text messages** labelled spam or not. SMS spam uses exactly the same tricks (prizes, "call now", "FREE!!!"), so the problem is identical. Our messages don't include sender addresses, so we can't build IP rules, but you'll build the other two: a keyword list and a blocklist.

> 📖 **Reading along?** This story follows the spam filter example that opens Géron, *Hands-On Machine Learning*, Chapter 1 ("The Machine Learning Landscape"). Géron uses spam to explain *why* machine learning exists at all. Look out for 📖 notes connecting what you're doing to the book.

Grab a coffee. This takes about 90 minutes.

### A tiny timeline of the spam war

| When | What happened |
|---|---|
| **1978** | The first known spam email: a marketing message sent to a few hundred users of ARPANET, the internet's ancestor. |
| **Late 1990s** | Spam explodes. Filters are hand-written keyword lists, blocklists and IP rules. *(← you are here)* |
| **1998** | Researchers publish the idea of filtering junk mail with Bayesian statistics, a way of calculating probabilities from word counts. |
| **2002** | Paul Graham publishes the essay *"A Plan for Spam"*, and Bayesian filtering goes mainstream. |
| **Today** | Email services use machine learning models trained on billions of messages. |

Keep this timeline in mind. By the end of the notebook, you'll have travelled the whole thing.

---
## 1 · First, what are we actually trying to do?

Let's put the job in plain terms. A message comes in. We want to stick one of two labels on it: **ham** (a normal message, which we'll call `0`) or **spam** (call it `1`). In maths, we're looking for a function

$$f: \text{message} \rightarrow \{0, 1\}$$

that gets the label right as often as possible. In Python, that's just a function that takes some text and returns `0` or `1`. You'll write several of them today.

Now, here's the thing that makes this lab interesting. There are two completely different ways to get your hands on that function.

**The 1999 way (traditional programming):** *you* write it. You study spam, spot patterns, and turn them into `if` statements. You put in rules and data, and answers come out.

**The machine learning way:** you collect thousands of messages that are *already labelled*, and let an algorithm work out the rules for you. You put in data *and* the answers, and the rules come out. Formally, the algorithm searches through a family of possible functions $\mathcal{H}$ and picks the one that makes the fewest mistakes on your labelled examples:

$$\hat{f} = \arg\min_{f \in \mathcal{H}} \frac{1}{n} \sum_{i=1}^{n} L\big(f(x_i),\, y_i\big)$$

Don't panic if that looks scary. It just says: *"out of all the functions I could choose, pick the one with the smallest average error $L$ on my $n$ examples."* By the end of this notebook you'll have done this search by hand, painfully, and you'll see why we hand it to a computer.

Back in 1959, Arthur Samuel described machine learning as giving computers the ability to learn *without being explicitly programmed*. Tom Mitchell later made it precise: a program **learns** from experience **E** at some task **T**, measured by performance **P**, if its performance at T gets better with E. For us:

- **T** is sorting messages into spam and ham,
- **E** is a big pile of labelled messages (the **training set**),
- **P** is how well it does on messages it has never seen.

> 📖 Géron opens his book with exactly these two definitions, then uses a spam filter to make them concrete. Same idea here.

So why was writing the rules yourself so hard? Keep these five reasons in the back of your mind. You're about to bump into every single one:

1. **There are way too many possible messages.** Even if we only care about *which words appear*, a vocabulary of 10,000 words gives $2^{10{,}000}$ possible combinations. Nobody can think through all of them.
2. **You know more than you can say.** You can spot spam in half a second, but try writing down *exactly* how you do it.
3. **Rules trip over each other.** Fix one mistake and two new ones pop up somewhere else.
4. **Spammers fight back.** The moment you block "FREE", they start writing "FR33".
5. **It never ends.** Every new spam campaign means someone has to write, test and ship a new rule.

---
## 2 · Before you touch the keyboard

Take five minutes and think these through (with a friend, if you have one handy). Jot your answers in the cell below. There are no wrong answers yet; we're just warming up.

1. Write down five rules you'd use to spot spam. Now, for each rule, think of a perfectly *normal* message that would get caught by mistake.
2. Imagine a lazy "filter" that just labels **everything** as ham. About 13% of messages are spam. How accurate is it? Would you ship it?
3. Which mistake is worse: a real message from your mum ending up in the spam folder, or a spam message getting through? Would your answer be different for a bank trying to catch fraud?
4. A self-driving car needs to spot pedestrians. What would **T**, **E** and **P** be?

In [ ]:
# Your warm-up answers go here.
# Lines starting with # are "comments": Python ignores them, so you can write anything.
# 1.
# 2.
# 3.
# 4.

---
## 3 · Meet the data

Time to meet our pile of messages: 5,572 real text messages that someone already sorted into spam and ham by hand. It's called the **SMS Spam Collection**, and it's going to be our whole world for this notebook.

First, we load the tools we need. Python on its own can't do much with data, so we **import** libraries that other people have written:

- **pandas** handles tables of data (think of it as Excel, but in code),
- **numpy** does fast maths on lists of numbers,
- **matplotlib** draws charts,
- **scikit-learn** (imported as `sklearn`) is the machine learning toolkit we'll use at the end.

In [ ]:
# --- Step 1: import the libraries we need ---------------------------------
# "import X as Y" loads library X and gives it a short nickname Y,
# so we can type pd.something instead of pandas.something.
from pathlib import Path          # Path helps us work with file locations

import numpy as np                # numpy: fast maths on lists of numbers
import pandas as pd               # pandas: tables of data (like a spreadsheet)
import matplotlib.pyplot as plt   # matplotlib: drawing charts
from sklearn.model_selection import train_test_split   # splits data into two parts

# Show up to 120 characters of each message when we print a table,
# so long messages don't get cut off too early.
pd.set_option("display.max_colwidth", 120)

# --- Step 2: say where the data lives -------------------------------------
# If the file isn't on our computer yet, we'll download it from this web address.
DATA_URL = ("https://raw.githubusercontent.com/mohitgupta-1O1/"
            "Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv")

# Places on our computer where the file might already be saved.
LOCAL_CANDIDATES = [Path("data/spam.csv"), Path("../data/spam.csv")]


# --- Step 3: a function that loads the data -------------------------------
# "def" creates a function: a reusable block of code with a name.
def load_raw():
    """Load spam.csv from the data/ folder if it's there, otherwise download it."""
    # Check each possible location, one at a time.
    for path in LOCAL_CANDIDATES:
        if path.exists():
            print(f"Loading {path}")
            # read_csv turns a .csv file into a pandas table (called a DataFrame).
            # encoding="latin-1" tells pandas how the text characters are stored.
            return pd.read_csv(path, encoding="latin-1")

    # If we got here, no local copy was found, so download it instead.
    print("Local copy not found, downloading...")
    raw = pd.read_csv(DATA_URL, encoding="latin-1")

    # Save a copy so we don't have to download it again next time.
    for path in LOCAL_CANDIDATES:
        if path.parent.exists():               # does the data/ folder exist?
            raw.to_csv(path, index=False, encoding="latin-1")
            print(f"Saved a copy to {path}")
            break                              # stop after saving once
    return raw


# --- Step 4: actually load it and take a peek -----------------------------
raw = load_raw()
print(raw.shape)      # .shape tells us (number of rows, number of columns)
raw.head()            # .head() shows the first 5 rows

Wait. We expected two columns: the label (`v1`) and the message (`v2`). So why are there *five*?

Welcome to real data. Some messages contained commas that weren't handled properly when the file was saved, so pandas chopped them up and spilled the leftovers into those `Unnamed` columns. Let's count how many got mangled.

In [ ]:
# raw.columns is the list of column names. [2:] means "from position 2 onwards",
# which picks out the three unwanted Unnamed columns.
# (Python starts counting at 0, so position 2 is the third column.)
extra_cols = raw.columns[2:]

# .notna() is True wherever a cell is NOT empty.
# .any(axis=1) asks, for each row: "is ANY of these extra cells filled in?"
spilled = raw[extra_cols].notna().any(axis=1)

# Adding up True/False values counts the Trues (True counts as 1, False as 0).
print(f"Messages that spilled into extra columns: {spilled.sum()}")

# Show the first 3 broken rows.
raw[spilled].head(3)

Fifty broken messages. Not a disaster, but if we ignored them we'd be analysing chopped-up text. The fix: glue every piece back together with the comma that split it.

In [ ]:
# For every row, take all the text pieces (column 1 onwards), drop the empty
# ones, and join them back together with commas.
#   .iloc[:, 1:]    -> every row, columns from position 1 onwards
#   .apply(..., axis=1) -> run the function once for each row
#   lambda row: ... -> a tiny one-line function, written in place
text = raw.iloc[:, 1:].apply(lambda row: ",".join(row.dropna().astype(str)), axis=1)

# Build a clean table with just two columns: the label and the full message.
df = pd.DataFrame({"label": raw["v1"], "text": text})

# Computers prefer numbers to words, so add a column that is
# 1 for spam and 0 for ham.
#   (df["label"] == "spam") gives True/False for every row,
#   .astype(int) turns True into 1 and False into 0.
df["is_spam"] = (df["label"] == "spam").astype(int)

# Count how many messages have each label.
print(df["label"].value_counts())

# The average of a column of 0s and 1s is the share of 1s, i.e. the share of spam.
# The ":.1%" part formats the number as a percentage with 1 decimal place.
print(f"\nSpam share: {df['is_spam'].mean():.1%}")

So roughly one message in eight is spam. Keep that 13.4% in your head; it matters more than you'd think.

Now, the step every 1999 rule-writer started with: **actually read some spam.** Géron describes the traditional approach as a loop that begins exactly here: study what spam looks like, write a rule for each pattern you notice, test, and repeat. Let's do step one.

In [ ]:
print("--- Some SPAM ---")
# df[df.is_spam == 1] keeps only the rows where is_spam is 1.
# .sample(5, random_state=1) picks 5 random rows. Fixing random_state means
# we get the same 5 every time we run it, so everyone sees the same examples.
for t in df[df.is_spam == 1]["text"].sample(5, random_state=1):
    print("*", t)

print("\n--- Some HAM ---")      # \n prints a blank line first
for t in df[df.is_spam == 0]["text"].sample(5, random_state=1):
    print("*", t)

**Your turn:** before reading on, write down at least five things you notice about the spam. Words, symbols, numbers, the way it's written.

(If you see `å£` instead of `£`, that's an encoding hiccup in the file. The pound sign is still in there, just wearing a disguise.)

### One crucial thing before we write any rules

We're going to split the data in two. You'll study and build your rules using the **training set** only. The **test set** gets locked in a drawer until the very end.

Why the secrecy? Because the whole point of a spam filter is to handle messages it has *never seen before*. If you write rules while staring at every message, of course they'll work on those messages. The test set is our honest judge.

> 📖 Géron makes the same point in Chapter 1: the only way to know how well a model will cope with new cases is to actually try it on new cases. That's what the test set is for.

In [ ]:
# train_test_split shuffles the messages and splits them into two groups.
#   test_size=0.2           -> 20% go to the test set, 80% to the training set
#   stratify=df["is_spam"]  -> keep the same share of spam (13.4%) in both groups
#   random_state=42         -> shuffle the same way every time, so results repeat
train, test = train_test_split(df, test_size=0.2, stratify=df["is_spam"], random_state=42)

# len() counts rows. The ":," in {len(train):,} adds a thousands separator (4,457).
print(f"training set: {len(train):,} messages ({train.is_spam.mean():.1%} spam)")
print(f"test set:     {len(test):,} messages ({test.is_spam.mean():.1%} spam)  <- no peeking!")

---
## 4 · How do we keep score?

Before we build anything, we need a way to tell a good filter from a bad one. This is Mitchell's **P**, the performance measure.

The obvious choice is **accuracy**: what fraction of messages did we label correctly? But remember the lazy "everything is ham" filter from the warm-up? Hold that thought.

Every prediction our filter makes lands in one of four boxes:

| | **We said spam** | **We said ham** |
|---|---|---|
| **Really spam** | ✅ True Positive (TP) | ❌ False Negative (FN): spam slipped through |
| **Really ham** | ❌ False Positive (FP): a real message got binned | ✅ True Negative (TN) |

From those four counts we get everything we need:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \qquad
\text{Precision} = \frac{TP}{TP + FP} \qquad
\text{Recall} = \frac{TP}{TP + FN}$$

In plain English: **precision** asks *"when we shout spam!, how often are we right?"* and **recall** asks *"of all the spam out there, how much did we actually catch?"* Usually you can push one up only by letting the other slide, so we also track their balance, the **F1 score**:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Your turn:** finish the `evaluate` function below by filling in the four formulas. The four counts (`tp`, `fp`, `fn`, `tn`) are already worked out for you.

In [ ]:
def evaluate(rule_fn, data, name="rule"):
    """Run a spam filter on every message in `data` and print how well it did.

    rule_fn : a function that takes a message and returns 1 (spam) or 0 (ham)
    data    : a table of messages with a "text" and an "is_spam" column
    name    : a label to print next to the scores
    """
    # Run our filter on every single message. .apply calls rule_fn once per message
    # and collects all the answers (0s and 1s) into a column called preds.
    preds = data["text"].apply(rule_fn)

    # The true answers, so we can compare.
    y = data["is_spam"]

    # Count the four boxes. "&" means AND: both conditions must be True.
    tp = int(((preds == 1) & (y == 1)).sum())   # said spam, really spam   (good)
    fp = int(((preds == 1) & (y == 0)).sum())   # said spam, really ham    (bad)
    fn = int(((preds == 0) & (y == 1)).sum())   # said ham,  really spam   (bad)
    tn = int(((preds == 0) & (y == 0)).sum())   # said ham,  really ham    (good)

    # TODO: replace each 0.0 with the correct formula from the markdown above.
    # Careful: a filter that never says "spam" makes tp + fp equal 0, and dividing
    # by zero crashes Python. Hint: "x / y if y else 0.0" avoids that.
    accuracy  = 0.0
    precision = 0.0
    recall    = 0.0
    f1        = 0.0

    # Print everything on one line. ":.3f" means "show 3 decimal places",
    # and ":<22" means "pad with spaces to 22 characters" so the columns line up.
    print(f"{name:<22} acc={accuracy:.3f}  prec={precision:.3f}  rec={recall:.3f}  "
          f"f1={f1:.3f}  FP={fp:<4} FN={fn}")

    # Also hand the scores back as a dictionary, in case we want to use them later.
    return {"name": name, "accuracy": accuracy, "precision": precision,
            "recall": recall, "f1": f1, "fp": fp, "fn": fn}

Now let's give that lazy filter its moment in the spotlight. It's our **baseline**: if anything we build can't beat it, we've wasted our time.

In [ ]:
# The laziest possible filter: whatever the message, say "ham" (0).
def always_ham(text):
    return 0

# The semicolon at the end hides the returned dictionary,
# so we only see the line that evaluate() prints.
evaluate(always_ham, train, "always ham");

That's our bar. Time to build something.

---
## 5 · Attempt #1: the keyword list

The very first spam filters of the 1990s were keyword lists. Spot a suspicious word, block the message. (The classic example was "Viagra"; our text messages don't have much of that, so we'll pick a different word.)

What's the most obviously spammy word you saw in Section 3? For a lot of people, it's **free**. Let's start there. One rule, one `if`.

In [ ]:
def is_spam_v1(text):
    # .lower() turns the message into lowercase, so "FREE", "Free" and "free"
    # all look the same to our rule.
    # "free" in ... is True if the word appears anywhere in the message.
    if "free" in text.lower():
        return 1      # spam
    else:
        return 0      # ham

evaluate(is_spam_v1, train, "v1: 'free'");

Better than nothing! But look at that recall. And we're binning some real messages too. Which ones? Géron's loop says that after testing, you **analyse your errors**. So let's look at them.

**Your turn:** finish `show_errors` so it prints the messages our rule got wrong.

In [ ]:
def show_errors(rule_fn, data, kind="fp", n=5):
    """Print up to n mistakes: false positives (kind="fp") or false negatives (kind="fn")."""
    preds = data["text"].apply(rule_fn)          # the filter's answer for every message

    # TODO: build a "mask" (a column of True/False) that is True for the mistakes.
    #   False positive: preds is 1 but data["is_spam"] is 0
    #   False negative: preds is 0 but data["is_spam"] is 1
    # Then print the first n of those messages with:
    #   for t in data.loc[mask, "text"].head(n):
    #       print("*", t)
    pass   # "pass" means "do nothing"; delete it once you've written your code

show_errors(is_spam_v1, train, kind="fp")

Before we pile on more keywords, let's try the other weapon from 1999.

---
## 6 · Attempt #2: the blocklist

The second tool in the 1990s toolbox was the **blocklist**: a list of known spammers. If a message comes from someone on the list, block it.

We don't know who sent our messages, but spam texts nearly always contain a **phone number or short code** to call or text back (like 87121). That's the spammer's "address". So here's our blocklist: collect every long number that appears in the spam in our training set, and block any message that mentions one of them.

To find the numbers we'll use a **regular expression** (or *regex*), a mini-language for describing text patterns. The pattern `\d{5,}` means "a digit (`\d`), repeated 5 or more times (`{5,}`)".

In [ ]:
import re      # Python's built-in library for regular expressions

# Build the blocklist. A "set" is like a list, but it ignores duplicates,
# which is perfect here: we only need each number once.
blocklist = set()

# Loop over every SPAM message in the training set...
for t in train[train.is_spam == 1]["text"]:
    # re.findall returns a list of every match of the pattern in the text,
    # e.g. "Call 08712300220 or text 87121" -> ["08712300220", "87121"]
    numbers = re.findall(r"\d{5,}", t)
    blocklist.update(numbers)        # add them all to our set

print(f"Numbers on the blocklist: {len(blocklist)}")
print("A few examples:", sorted(blocklist)[:5])


def is_spam_blocklist(text):
    # Find every long number in this message...
    for number in re.findall(r"\d{5,}", text):
        # ...and if any of them is on our blocklist, it's spam.
        if number in blocklist:
            return 1
    return 0     # no blocked numbers found: let it through

evaluate(is_spam_blocklist, train, "blocklist");

Whoa. **Perfect precision**: not a single real message blocked. And it catches most of the spam too. Have we solved it already?

Hold that excitement. Ask yourself: where did every number on the list come from? We'll find out what that means in Section 9.

For now, let's get back to keywords, because spammers don't always include a number.

---
## 7 · Attempt #3: more keywords!

Let's grow a chain of `if` / `elif` checks, one for each pattern we spotted. **Target: get recall above 0.80** on the training set, without wrecking precision.

A tip: re-run `evaluate` after *every* rule you add, and jot down what happened. You'll want those notes later in the story.

In [ ]:
def is_spam_v2(text):
    t = text.lower()                 # lowercase once, then check against t

    # Python checks these one at a time from the top.
    # "elif" means "else if": only checked if every rule above it was False.
    # As soon as one rule matches, the function returns and stops.
    if "free" in t and "call" in t:  # "and": BOTH words must appear
        return 1
    elif "winner" in t or "won" in t:  # "or": EITHER word is enough
        return 1
    # TODO: keep adding "elif ...: return 1" rules here, one pattern at a time
    else:                            # nothing matched, so call it ham
        return 0

evaluate(is_spam_v2, train, "v2: if/elif chain");

How did it go? A few things to chew on:

- Did you reach 0.80 recall? How many rules did it take?
- `"won" in t` also matches *won't*. What other words did your rules catch by accident?
- Did any rule make precision *worse*? Why does that keep happening?

You might be noticing the list getting long and fiddly. Paul Graham ran into exactly this when he tried the rule-based approach himself: his single rule for the word "click" caught about 80% of his spam, but every extra percent after that got harder, and stricter rules started catching more real mail. Géron says the same: the traditional approach ends up as a long list of complicated rules that's hard to maintain.

But before we give up, there's one more trick worth trying.

---
## 8 · Attempt #4: let the clues vote

Instead of letting any single clue convict a message, what if each clue gets a **score**? Strong clues ("prize", "claim") get a big score. Weak ones ("call") get a small one. Add up the scores, and only flag the message if the total crosses a **threshold**.

In maths, that looks like this:

$$\text{score}(x) = \sum_{j} w_j \, \phi_j(x), \qquad
\hat{y} = \begin{cases} 1 & \text{if } \text{score}(x) \ge \theta \\ 0 & \text{otherwise} \end{cases}$$

That's a lot of Greek, so here it is in English:

- $\phi_j(x)$ is 1 if clue number $j$ is in the message, and 0 if it isn't,
- $w_j$ is how many points that clue is worth (its **weight**),
- $\sum$ means "add them all up",
- $\theta$ (theta) is the **threshold**: score at least this much, and you're spam.

**Your turn:** play with the weights in `KEYWORDS`, the digit bonus, and `THRESHOLD`. Try to get the best **F1** you can on the training set. And count how many tries it takes you. Seriously, keep count.

In [ ]:
# A dictionary maps each clue (a "key") to how many points it's worth (a "value").
KEYWORDS = {
    "free": 1.0, "win": 1.5, "prize": 2.0, "claim": 2.0,
    # TODO: add more clues here, e.g. "urgent": 1.5, and tune the weights
}
LONG_NUMBER_BONUS = 0.0   # TODO: try giving extra points for long numbers like 87121
THRESHOLD = 2.0           # TODO: tune. Score at least this much and we call it spam


def spam_score(text):
    """Add up the points for every clue found in the message."""
    t = text.lower()

    # KEYWORDS.items() gives us each (word, weight) pair in turn.
    # This line adds up the weight of every word that appears in the message.
    # It's a short way of writing:
    #     score = 0
    #     for word, weight in KEYWORDS.items():
    #         if word in t:
    #             score = score + weight
    score = sum(weight for word, weight in KEYWORDS.items() if word in t)

    # re.search looks for the pattern anywhere in the text.
    # It returns a match if it finds one, or None if it doesn't.
    if re.search(r"\d{5,}", t):          # 5 or more digits in a row?
        score += LONG_NUMBER_BONUS       # "+=" means "add to the existing value"
    return score


def is_spam_v3(text):
    # One line version of: if the score is high enough, return 1, otherwise 0.
    return 1 if spam_score(text) >= THRESHOLD else 0


evaluate(is_spam_v3, train, "v3: weighted score");

Now stop and look at that formula one more time. Clues, weights, add them up, compare to a threshold.

> ### 💡 Plot twist
>
> **You just built a machine learning model.** That formula is a *linear classifier*, one of the most widely used models in the whole field.
>
> The only thing that isn't "machine learning" about it is *how you found the weights*: by guessing, checking the score, and guessing again. A learning algorithm does exactly that, except it's systematic, it's fast, and it can juggle thousands of weights at once instead of twelve.
>
> **Machine learning doesn't throw away the idea of rules. It replaces *you* as the person who tunes them.**

So, how many tries did tuning take you? Are you *sure* your weights are the best possible? Now imagine doing this for 5,000 words instead of 12. Yeah.

> 📖 Géron calls the part of an ML system that makes predictions a **model**, and the process of finding its best settings **training**. Without realising it, you've been training a model by hand.

---
## 9 · The moment of truth

Okay. Time to open the drawer. Let's run every filter on the **test set**, the messages you've never laid eyes on.

In [ ]:
# A list of (function, name) pairs: every filter we've built so far.
filters = [
    (always_ham,        "always ham"),
    (is_spam_v1,        "v1 keyword"),
    (is_spam_blocklist, "blocklist"),
    (is_spam_v2,        "v2 if/elif"),
    (is_spam_v3,        "v3 weighted"),
]

# For each filter, score it on the training set, then on the test set.
for fn, name in filters:
    evaluate(fn, train, f"{name:<12} TRAIN")
    evaluate(fn, test,  f"{name:<12} TEST")
    print()        # a blank line between filters, to keep it readable

Compare each filter's training and test scores. Look especially at the **blocklist**.

Here's why we care. The longer you stare at a set of messages and tweak your rules to fit them, the more your rules end up fitting *those particular messages*, quirks and all, rather than spam in general. That's called **overfitting**, and it's one of the biggest traps in machine learning. As you're discovering, humans fall into it too.

> 📖 Overfitting gets its own section in Géron's Chapter 1, under the main challenges of machine learning. You've just met it in the wild.

---
## 10 · The spammers strike back

A few weeks after you ship v3, spam starts sneaking through again. The spammers noticed what was getting blocked, and they adapted. Here are two of their new messages, plus two perfectly innocent ones from real customers.

In [ ]:
# Each item is a pair: (the true label, the message).
adversarial = [
    ("spam", "Congratulations! U have W0N a FR33 pr!ze. Reply Y to cl@im"),
    ("spam", "W-I-N-N-E-R!! Exclusive reward waiting 4 u"),
    ("ham",  "Hey, are you free to call me after the lecture?"),
    ("ham",  "Did you claim your expenses from the conference?"),
]

# "for truth, msg in adversarial" unpacks each pair into two variables.
for truth, msg in adversarial:
    pred = "spam" if is_spam_v3(msg) else "ham"   # what our filter says
    mark = "✓" if pred == truth else "✗"          # right or wrong?
    print(f"{mark} truth={truth:<4} pred={pred:<4} | {msg}")

Ouch. Both spam messages walk straight through, and a customer asking about their expenses gets binned.

**Your turn:**
1. Patch v3 so it catches both spam messages *without* flagging the innocent ones.
2. Now switch sides. Put on your spammer hat and write **three new messages** that sneak past your patched filter.
3. How long could this go on? Who's going to win?

In [ ]:
# Your patched rules, and your three sneaky new spam messages, go here.

---
## 11 · Stepping back: more rules, better filter?

Let's zoom out. Below, we add keyword rules one at a time and measure the filter after each addition. If you wrote your own rules in Section 7, swap them into `RULES` in the order you added them.

In [ ]:
# A list of (name, rule) pairs. Each rule is a tiny "lambda" function that takes
# the lowercased message t and returns True if its clue is present.
RULES = [
    ("free",       lambda t: "free" in t),
    ("winner/won", lambda t: "winner" in t or "won" in t),
    # TODO: add your own rules, in the order you added them in Section 7
]

In [ ]:
def first_k_rules(k):
    """Build a filter that uses only the first k rules from RULES.

    The filter says spam if ANY of those rules fires.
    (This is a function that builds and returns another function!)
    """
    def rule_fn(text):
        t = text.lower()
        # RULES[:k] means "the first k items".
        # any(...) is True if at least one rule returns True.
        return 1 if any(check(t) for _, check in RULES[:k]) else 0
    return rule_fn


import contextlib, io     # used only to hide evaluate()'s printing below

log = []                  # we'll collect one row of results per step here

# range(1, len(RULES) + 1) counts 1, 2, 3, ... up to the number of rules.
for k in range(1, len(RULES) + 1):
    # Run evaluate() quietly: we'll show the results as a table instead.
    with contextlib.redirect_stdout(io.StringIO()):
        tr = evaluate(first_k_rules(k), train)
        te = evaluate(first_k_rules(k), test)

    # Save this step's results. RULES[k - 1][0] is the name of the rule just added.
    log.append({"rules": k, "added": RULES[k - 1][0],
                "train_f1": tr["f1"], "test_f1": te["f1"],
                "train_precision": tr["precision"], "train_recall": tr["recall"]})

# Turn the list of results into a table and show it, rounded to 3 decimals.
log = pd.DataFrame(log)
display(log.round(3))

# --- Draw the chart ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))    # an empty chart, 8 wide by 4 tall
ax.plot(log["rules"], log["train_precision"], marker="o", label="train precision")
ax.plot(log["rules"], log["train_recall"], marker="o", label="train recall")
ax.plot(log["rules"], log["train_f1"], marker="o", linewidth=2.5, label="train F1")
ax.plot(log["rules"], log["test_f1"], marker="s", linestyle="--", label="test F1")
ax.set_xlabel("number of rules")
ax.set_ylabel("score")
ax.set_ylim(0, 1)                          # scores always sit between 0 and 1
ax.set_title("What happens as you keep adding rules?")
ax.legend()                                # the box that says which line is which
plt.tight_layout()
plt.show()

**Take a moment to reflect** (150–200 words): using your table and chart, what would it take to keep this filter working for the next five years? Think about new spam campaigns, new slang, the arms race from Section 10, and who would be stuck maintaining all those rules and blocklists.

In [ ]:
# Your reflection goes here (or change this cell to a markdown cell).

---
## 12 · Fast-forward to 2002: enter machine learning

By the early 2000s, programmers everywhere were exactly where you are now: buried under keyword lists, blocklists that went stale, and spammers who changed their spelling every week.

Then, in **August 2002**, the programmer and essayist **Paul Graham** published an essay called *"A Plan for Spam"*. His idea was simple and powerful. Stop trying to write rules. Instead, take a pile of your own spam and a pile of your own good mail, **count how often every word appears in each pile**, and use those counts to calculate the **probability** that a new message is spam.

This approach is called **Bayesian filtering**, and the algorithm behind it is known as **Naive Bayes**. Researchers had already proposed it a few years earlier, but Graham's essay made it famous. On his own email, he reported that his filter missed fewer than 5 in every 1,000 spam messages, while wrongly flagging **zero** real ones. Within a couple of years, Bayesian filters were being built into popular email programs and spam-filtering tools, and **supervised machine learning** (learning from labelled examples) had become part of everyday email security.

So let's do exactly what Graham did. We'll hand the computer our labelled training set and say: *you work out which words matter, and how much.* Here it is in four lines. How it actually works is the next notebook; for now, just watch the numbers.

In [ ]:
# CountVectorizer turns each message into word counts
#   ("free prize free" -> free: 2, prize: 1).
# MultinomialNB is Naive Bayes: the same family of algorithm Paul Graham made famous.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# make_pipeline chains the two steps: first count the words, then classify.
model = make_pipeline(CountVectorizer(), MultinomialNB())

# .fit() is where the learning happens: the model studies every training
# message alongside its correct label, and works out its own weights.
model.fit(train["text"], train["is_spam"])

# How many words did it learn a weight for? (One per word in its vocabulary.)
n_weights = len(model.named_steps["countvectorizer"].vocabulary_)
print(f"Your v3 filter: {len(KEYWORDS)} hand-picked keyword weights")
print(f"This model:     {n_weights:,} weights, learned in under a second\n")

# Wrap the model in a function that works just like our hand-written filters,
# so evaluate() can score it the same way.
# model.predict([...]) expects a list of messages and returns a list of answers;
# [0] takes the first (and only) answer.
learned = lambda text: int(model.predict([text])[0])

evaluate(is_spam_v3, test, "v3 (hand-tuned) TEST")
evaluate(learned,    test, "Naive Bayes TEST");

Thousands of weights, learned in less time than it took you to tune your first threshold. And it catches more spam than your best hand-tuned effort.

But how does it do against those sneaky spammers from Section 10?

In [ ]:
# Same test as Section 10, but now using the learned model.
for truth, msg in adversarial:
    pred = "spam" if learned(msg) else "ham"
    mark = "✓" if pred == truth else "✗"
    print(f"{mark} truth={truth:<4} pred={pred:<4} | {msg}")

Not perfect, and that's an important lesson: **machine learning isn't magic.** It's still making a mistake. But think about what happens next week, when the spammers change tactics again. With your rules, someone has to sit down, study the new spam, write new rules, update the blocklist and re-tune every weight. With the model, you label a few hundred new examples and run `fit` again. Géron makes exactly this point: an ML-based filter can notice that new words have started showing up in spam and adapt, without anyone rewriting the code.

### One last twist: the model can teach *you*

Remember Section 3, where you squinted at spam trying to spot patterns? The model did the same thing, across every single message, and it kept score. So let's ask it: *which words did you decide are the strongest signs of spam?*

In [ ]:
# Pull the two steps back out of the pipeline so we can look inside them.
vectorizer = model.named_steps["countvectorizer"]
nb = model.named_steps["multinomialnb"]

# Every word the model knows, in the same order as its internal weights.
words = vectorizer.get_feature_names_out()

# How many times each word appears across the whole training set.
counts = np.asarray(vectorizer.transform(train["text"]).sum(axis=0)).ravel()

# For each word, how much more likely it is to show up in spam than in ham.
# feature_log_prob_[1] is spam, [0] is ham. These are on a "log" scale, so
# subtracting them compares the two: a bigger number means "more spammy".
spamminess = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]

# Only consider words that appear at least 20 times. Very rare words give
# noisy, unreliable results.
common = counts >= 20

# np.argsort sorts from smallest to largest; [::-1] flips it to largest first.
ranked = np.argsort(spamminess[common])[::-1]
top_words = words[common][ranked][:20]     # keep the top 20

print("Top 20 spam signals the model found on its own:\n")
print(", ".join(top_words))

Compare that list with the five patterns you wrote down in Section 3. How many did you spot? Which ones did the model find that you missed?

> 📖 Géron lists this as one of the underrated reasons to use ML: you can inspect what a trained model learned and discover patterns humans hadn't noticed. Digging through data like this is sometimes called **data mining**.

---
## Where the story goes next

Let's recap. You started in 1999 and fought spam the way everyone did back then. You built a keyword list, and it was too blunt. You built a blocklist, and it only knew spammers it had already seen. You added more and more rules, discovered that clues need weights, tuned those weights by hand, and got ambushed by spammers. Then you jumped to 2002, did what Paul Graham did, and a four-line model beat you, adapted more easily, and even taught you something about your own data.

That's machine learning in a nutshell: **instead of writing the rules, you write the code that finds the rules.**

The obvious next question is *how*. How did Naive Bayes actually come up with those 7,774 weights? That's where we're headed next.

---
## Want to go further?

1. **Smarter matching.** Rewrite v2 using `re.search(r"\bfree\b", t)` so "free" only matches as a whole word (`\b` means "word boundary"). Does precision improve? Is the code easier or harder to read?
2. **Those duplicates.** Run `df["text"].duplicated().sum()`. Remove duplicates *before* splitting (look up `df.drop_duplicates`), then redo Section 9 and Section 12. What happens to the test scores, and why does that matter?
3. **Are rules ever the right answer?** Think of two real situations where you'd pick hand-written rules over ML. (Hints: what if you have almost no data? What if the law says you must explain every decision?)

## Further reading

- 📖 **Géron, A.** *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (or the earlier *...with Scikit-Learn and TensorFlow*), O'Reilly. **Chapter 1, "The Machine Learning Landscape"**: read "What Is Machine Learning?", "Why Use Machine Learning?" and "Main Challenges of Machine Learning" alongside this notebook. Companion code: [github.com/ageron/handson-ml3](https://github.com/ageron/handson-ml3)
- Graham, P. (2002). [*A Plan for Spam.*](http://www.paulgraham.com/spam.html) The essay from Section 12. Short, readable, and a great piece of history.
- Sahami, M., Dumais, S., Heckerman, D., & Horvitz, E. (1998). *A Bayesian Approach to Filtering Junk E-Mail.* AAAI Workshop on Learning for Text Categorization. One of the research papers that proposed Bayesian spam filtering before Graham.
- Samuel, A. L. (1959). *Some Studies in Machine Learning Using the Game of Checkers.* IBM Journal of Research and Development.
- Mitchell, T. M. (1997). *Machine Learning*, Chapter 1. McGraw-Hill.
- Domingos, P. (2012). *A Few Useful Things to Know About Machine Learning.* Communications of the ACM, 55(10).
- Almeida, T. A., Gómez Hidalgo, J. M., & Yamakami, A. (2011). *Contributions to the Study of SMS Spam Filtering: New Collection and Results.* ACM DocEng 2011. The paper behind this dataset.
- [scikit-learn: Working With Text Data](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)